In [1]:
import warnings
warnings.filterwarnings("ignore")
import ast
import shutil
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Bidirectional, Conv1D, MaxPooling1D, Flatten, Embedding, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.regularizers import l2
from sklearn.metrics import accuracy_score, f1_score, classification_report, recall_score, precision_score
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
from scikeras.wrappers import KerasClassifier, KerasRegressor

from keras_tuner import RandomSearch
import keras_tuner as kt
import tensorflow as tf
import tensorflow_decision_forests as tfdf
import os
from gensim.models import FastText

2024-08-23 11:08:49.207493: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-08-23 11:08:49.207940: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-23 11:08:49.210249: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-23 11:08:49.216829: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-23 11:08:49.228545: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 

In [2]:
def get_document_vector1(doc, model):
    vectors = [model.wv[word] for word in doc if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

## Khởi tạo các mô hình

In [3]:
def lstm_model(hp, embedding_matrix):
    lstm_units=hp.Int('lstm_units_1', min_value=12, max_value=48, step=8)
    dropout_rate=0.2
    vocab_size=10000
    embedding_dim=50
    max_length=50

    model = Sequential()
    model.add(Embedding(input_dim=embedding_matrix.shape[0], 
                        output_dim=embedding_dim,
                        weights=[embedding_matrix], 
                        input_length=max_length, trainable=False))
    model.add(LSTM(lstm_units, return_sequences=False))
    model.add(Flatten())
    model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [4]:
def bilstm_model(hp, embedding_matrix):
    lstm_units_1=hp.Int('lstm_units_1', min_value=12, max_value=48, step=4)
    lstm_units_2=hp.Int('lstm_units_1', min_value=12, max_value=44, step=4)
    dropout_rate=0.2
    vocab_size=10000
    embedding_dim=50
    max_length=50

    model = Sequential()
    model.add(Embedding(input_dim=embedding_matrix.shape[0], 
                        output_dim=embedding_dim,
                        weights=[embedding_matrix], 
                        input_length=max_length, trainable=False))
    
    model.add(Bidirectional(LSTM(lstm_units_1, return_sequences=True)))
    model.add(Bidirectional(LSTM(lstm_units_2)))
    model.add(Flatten())
    model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [5]:
def cnn_lstm_model(hp, embedding_matrix):
    lstm_units=hp.Int('lstm_units_1', min_value=12, max_value=48, step=4)
    conv_filters=hp.Int('conv_filters_1', min_value=12, max_value=64, step=8)
    kernel_size=3
    dropout_rate=0.2
    vocab_size=10000
    embedding_dim=50
    max_length=50


    model = Sequential()
    model.add(Embedding(input_dim=embedding_matrix.shape[0], 
                        output_dim=embedding_dim,
                        weights=[embedding_matrix], 
                        input_length=max_length, trainable=False))
    
    model.add(Conv1D(filters=conv_filters, kernel_size=kernel_size, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(LSTM(lstm_units))
    model.add(Flatten())
    model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [6]:
def ann_model(hp, embedding_matrix):
    dense_units_1=hp.Int('dense_units_1', min_value=4, max_value=25, step=4)
    dense_units_2=hp.Int('dense_units_2', min_value=4, max_value=25, step=4)
    dropout_rate=0.2
    vocab_size=10000
    embedding_dim=50
    max_length=50
    
    model = Sequential()
    model.add(Embedding(input_dim=embedding_matrix.shape[0], 
                        output_dim=embedding_dim,
                        weights=[embedding_matrix], 
                        input_length=max_length, trainable=False))
    
    model.add(Dense(dense_units_1, activation='relu'))
    model.add(Dense(dense_units_2, activation='relu'))
    model.add(Dropout(dropout_rate))
    model.add(Flatten())
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

## Build mô hình và in kết quả

In [7]:
def buildDL_fast(X_train, X_test, y_train, y_test):
    
    model_fast = FastText(sentences=X_train, vector_size=50, window=8, min_count=1, workers=4)

    #X_train = np.array([get_document_vector1(doc, model_fast) for doc in X_train])
    #X_test = np.array([get_document_vector1(doc, model_fast) for doc in X_test])
    
    #X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
    #X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(X_train)

    max_length=50
    vocab_size = len(tokenizer.word_index) + 1

    X_train_seq = tokenizer.texts_to_sequences(X_train)
    X_test_seq = tokenizer.texts_to_sequences(X_test)

    X_train_pad = pad_sequences(X_train_seq, maxlen=max_length)
    X_test_pad = pad_sequences(X_test_seq, maxlen=max_length)

    
    embedding_dim = model_fast.vector_size
    embedding_matrix = np.zeros((vocab_size, embedding_dim))

    

    for word, i in tokenizer.word_index.items():
        if word in model_fast.wv:
            embedding_matrix[i] = model_fast.wv[word]
        else:
            embedding_matrix[i] = np.zeros(embedding_dim)

    models = {
        'LSTM': lstm_model,
        'BiLSTM': bilstm_model,
        'CNN-LSTM': cnn_lstm_model,
        'ANN': ann_model
    }

    print(embedding_matrix.shape)
    res = []
    for key in models.keys():
        if os.path.exists('embedding_tuner_text') and os.path.isdir('embedding_tuner_text'):
            print("Directory exists.")
            shutil.rmtree('embedding_tuner_text')
        # else:
        #     print("Directory does not exist.")

        if os.path.exists('embedding_tuning_text') and os.path.isdir('embedding_tuning_text'):
            print("Directory exists.")
            shutil.rmtree('embedding_tuning_text')
        # else:
        #     print("Directory does not exist.")
        model = models[key]
        

        tuner = RandomSearch(
            lambda hp: model(hp, embedding_matrix=embedding_matrix),
            objective='val_accuracy',
            max_trials=10,
            executions_per_trial=1,
            directory='embedding_tuner_text',
            project_name='embedding_tuning_text'
        )

        tuner.search(X_train_pad, y_train, epochs=10, validation_data=(X_test_pad, y_test))

        best_model = tuner.get_best_models(num_models=1)[0]
        y_pred = best_model.predict(X_test_pad)
        y_pred = (y_pred > 0.5).astype(int).flatten()
        
        result = [key, accuracy_score(y_test, y_pred),
                  precision_score(y_test, y_pred),
                  recall_score(y_test, y_pred),
                  f1_score(y_test, y_pred)]
        res.append(result)

        print(f"Model {key} done !")
    return pd.DataFrame(res, columns=["Model", "Accuracy", "Precision", "Recall", "F1 score"])
    

In [8]:
def sentimentML_evaluate(data_train, data_test, typeData):
    origin_train_data = data_train[data_train["origin"] == 0]
    augmented_train_data = data_train[data_train["origin"] == 1]

    origin_test_data = data_test[data_test["origin"] == 0]
    augmented_test_data = data_test[data_test["origin"] == 1]
    
    X_train_origin = origin_train_data["Words"]
    X_test_origin = origin_test_data["Words"]

    X_train_augmented = augmented_train_data["Words"]
    X_test_augmented = augmented_test_data["Words"]
    
    le = LabelEncoder()
    y_train_origin = pd.DataFrame(le.fit_transform(origin_train_data.iloc[:, -1]))
    y_test_origin = pd.DataFrame(le.transform(origin_test_data.iloc[:, -1]))

    y_train_augmented = pd.DataFrame(le.transform(augmented_train_data.iloc[:, -1]))
    y_test_augmented = pd.DataFrame(le.transform(augmented_test_data.iloc[:, -1]))

    print("Basic origin - origin")
    res = buildDL_fast(X_train_origin, X_test_origin, y_train_origin, y_test_origin)
    res.to_csv(f'../../data/result/Text/{typeData}-oo.csv')

    print("Augmented - origin")
    res = buildDL_fast(pd.concat((X_train_origin, X_train_augmented), axis=0), X_test_origin, 
                      pd.concat((y_train_origin, y_train_augmented), axis=0), y_test_origin)
    res.to_csv(f'../../data/result/Text/{typeData}-ao.csv')

    # print("Augmented - augmented")
    # res = buildDL_fast(pd.concat((X_train_origin, X_train_augmented), axis=0), pd.concat((X_test_origin, X_test_augmented), axis=0), 
    #                   pd.concat((y_train_origin, y_train_augmented), axis=0), pd.concat((y_test_origin, y_test_augmented), axis=0))
    # res.to_csv(f'../../data/result/Text/{typeData}-aa.csv')

In [9]:
near_train = pd.read_csv(f"../../data/cleaned/Text/Near_processed_train.csv")[["Words", "origin", "Near"]]
near_test = pd.read_csv(f"../../data/cleaned/Text/Near_processed_test.csv")[["Words", "origin", "Near"]]

mid_train = pd.read_csv(f"../../data/cleaned/Text/Mid_processed_train.csv")[["Words", "origin", "Mid"]]
mid_test = pd.read_csv(f"../../data/cleaned/Text/Mid_processed_test.csv")[["Words", "origin", "Mid"]]

far_train = pd.read_csv(f"../../data/cleaned/Text/Far_processed_train.csv")[["Words", "origin", "Far"]]
far_test = pd.read_csv(f"../../data/cleaned/Text/Far_processed_test.csv")[["Words", "origin", "Far"]]

potential_train = pd.read_csv(f"../../data/cleaned/Text/Potential_processed_train.csv")[["Words", "origin", "Potential"]]
potential_test = pd.read_csv(f"../../data/cleaned/Text/Potential_processed_test.csv")[["Words", "origin", "Potential"]]

In [14]:
sentimentML_evaluate(mid_train, mid_test, 'Mid')

Trial 10 Complete [00h 00m 08s]
val_accuracy: 0.5215053558349609

Best val_accuracy So Far: 0.5537634491920471
Total elapsed time: 00h 01m 25s
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
Model ANN done !


In [11]:

sentimentML_evaluate(near_train, near_test, 'Near')

Trial 10 Complete [00h 00m 04s]
val_accuracy: 0.5752688050270081

Best val_accuracy So Far: 0.5860214829444885
Total elapsed time: 00h 00m 41s
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
Model ANN done !


In [12]:

sentimentML_evaluate(far_train, far_test, 'Far')

Trial 10 Complete [00h 00m 06s]
val_accuracy: 0.5645161271095276

Best val_accuracy So Far: 0.5752688050270081
Total elapsed time: 00h 00m 56s
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
Model ANN done !


In [13]:

sentimentML_evaluate(potential_train, potential_test, 'Potential')

Trial 10 Complete [00h 00m 07s]
val_accuracy: 0.5967742204666138

Best val_accuracy So Far: 0.602150559425354
Total elapsed time: 00h 01m 10s
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
Model ANN done !
